# 🎙️ RVC v2 — English TTS with Your Own Voice

> Run this notebook on **Google Colab with a free T4 GPU**.

## 📋 Workflow
```
Step 1: Install environment
Step 2: Upload your voice recordings
Step 3: Preprocess (slice + feature extraction)
Step 4: Train the model (10~30 min)
Step 5: Type any English text → hear it in your voice!
```

## ✅ Before You Start
- Go to **Runtime → Change runtime type → T4 GPU** (required!)
- Prepare voice recordings in WAV or MP3 format (5~30 min recommended)
- Connecting Google Drive is recommended to save your trained model


---
## 🔧 Step 1: Check GPU & Install Environment

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\n✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ No GPU found! Go to Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Mount Google Drive (to save trained model)
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

In [ ]:
# Clone RVC repo & install dependencies
import os

if not os.path.exists('/content/Retrieval-based-Voice-Conversion-WebUI'):
    print("📦 Cloning RVC repository...")
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git

%cd /content/Retrieval-based-Voice-Conversion-WebUI

print("📦 Installing required packages... (3~5 min)")
!pip install -q -r requirements.txt

# Additional dependencies (updated versions)
!pip install -q faiss-gpu fairseq gradio==3.50.0 torchcrepe
!apt-get install -q -y ffmpeg

print("✅ Installation complete!")

In [ ]:
# Download pretrained base models
import os

%cd /content/Retrieval-based-Voice-Conversion-WebUI

os.makedirs('assets/hubert', exist_ok=True)
os.makedirs('assets/rmvpe', exist_ok=True)
os.makedirs('assets/pretrained_v2', exist_ok=True)

print("📥 Downloading HuBERT model...")
!wget -q -O assets/hubert/hubert_base.pt \
  https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt

print("📥 Downloading RMVPE pitch model...")
!wget -q -O assets/rmvpe/rmvpe.pt \
  https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt

print("📥 Downloading pretrained v2 weights...")
!wget -q -O assets/pretrained_v2/f0D40k.pth \
  https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth
!wget -q -O assets/pretrained_v2/f0G40k.pth \
  https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth

print("✅ All models downloaded!")

---
## 🎤 Step 2: Upload Your Voice Recordings

### Recording Tips 📌
- **Length**: Minimum 5 min, recommended 10~30 min (longer = better quality)
- **Environment**: Quiet room, no echo
- **Content**: Read diverse English sentences naturally — mix short, long, questions, exclamations
- **Format**: WAV (recommended) or MP3
- **Sample rate**: 44100 Hz or 48000 Hz

> 💡 Multiple files are fine — they will all be merged into one dataset

In [ ]:
# Upload voice recording files
from google.colab import files
import os, shutil

MODEL_NAME  = "my_voice"  # <- Change this to your preferred model name
DATASET_DIR = f"/content/Retrieval-based-Voice-Conversion-WebUI/dataset/{MODEL_NAME}"
os.makedirs(DATASET_DIR, exist_ok=True)

print("📂 Please upload your recording files (WAV/MP3)...")
uploaded = files.upload()

for filename, data in uploaded.items():
    dest = os.path.join(DATASET_DIR, filename)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f"  ✅ Saved: {filename} ({len(data)/1024/1024:.1f} MB)")

print(f"\n✅ {len(uploaded)} file(s) uploaded")
print(f"📁 Dataset path: {DATASET_DIR}")

---
## ✂️ Step 3: Preprocess (Slice + Feature Extraction)

In [ ]:
# Training config — re-declared here so this cell is safe to re-run
MODEL_NAME  = "my_voice"   # Must match the name used in Step 2
SAMPLE_RATE = 40000        # 40000 or 48000
N_CPU       = 4            # Number of CPU cores
DATASET_DIR = f"/content/Retrieval-based-Voice-Conversion-WebUI/dataset/{MODEL_NAME}"

print(f"Model name  : {MODEL_NAME}")
print(f"Sample rate : {SAMPLE_RATE}")
print(f"Dataset path: {DATASET_DIR}")

In [ ]:
%cd /content/Retrieval-based-Voice-Conversion-WebUI

# 1. Slice audio into 3~10 second chunks
print("✂️ Slicing audio...")
!python trainset_preprocess_pipeline_print.py \
  "{DATASET_DIR}" {SAMPLE_RATE} {N_CPU} \
  "logs/{MODEL_NAME}" True

print("\n✅ Audio slicing complete!")

In [ ]:
# 2. Extract features (HuBERT embeddings + pitch)
print("🔍 Extracting features... (5~10 min)")
!python extract_f0_print.py \
  "logs/{MODEL_NAME}" {N_CPU} "rmvpe"

!python extract_feature_print.py \
  cuda:0 1 0 0 "logs/{MODEL_NAME}" v2

print("\n✅ Feature extraction complete!")

---
## 🏋️ Step 4: Train the Model

| Data length | Recommended epochs | Estimated time |
|-------------|-------------------|----------------|
| Under 5 min | 100~200 | 5~10 min |
| 10~20 min   | 200~300 | 15~25 min |
| 30+ min     | 300~500 | 30~60 min |

In [ ]:
import os

# Re-declare variables (safe to re-run)
MODEL_NAME       = "my_voice"  # <- Must match Step 2
SAMPLE_RATE      = 40000
N_CPU            = 4

# Training hyperparameters
TOTAL_EPOCH      = 200  # <- Adjust based on your data length
BATCH_SIZE       = 7    # Optimal for T4 GPU
SAVE_EVERY_EPOCH = 50   # Save a checkpoint every 50 epochs

LOG_DIR = f"logs/{MODEL_NAME}"
os.makedirs(LOG_DIR, exist_ok=True)

# Build filelist (written as a .py file to avoid quote conflicts)
script_lines = [
    'import os',
    f'log_dir    = "{LOG_DIR}"',
    'gt_wav_dir = os.path.join(log_dir, "0_gt_wavs")',
    'feat_dir   = os.path.join(log_dir, "3_feature256")',
    'f0_dir     = os.path.join(log_dir, "2a_f0")',
    'nsf_dir    = os.path.join(log_dir, "2b-f0nsf")',
    'out_path   = os.path.join(log_dir, "filelist.txt")',
    'lines = []',
    'for fname in os.listdir(gt_wav_dir):',
    '    if fname.endswith(".wav"):',
    '        name = fname[:-4]',
    '        wav  = os.path.join(gt_wav_dir, fname)',
    '        feat = os.path.join(feat_dir,   name + ".npy")',
    '        f0   = os.path.join(f0_dir,     name + ".wav.npy")',
    '        nsf  = os.path.join(nsf_dir,    name + ".wav.npy")',
    '        if os.path.exists(feat) and os.path.exists(f0) and os.path.exists(nsf):',
    '            lines.append(wav + "|" + feat + "|" + f0 + "|" + nsf)',
    'with open(out_path, "w") as fp:',
    '    fp.write("\\n".join(lines))',
    'print("Filelist created:", len(lines), "entries")',
]
with open('/tmp/make_filelist.py', 'w') as fp:
    fp.write('\n'.join(script_lines))

!python3 /tmp/make_filelist.py

print(f"\n🏋️ Starting training ({TOTAL_EPOCH} epochs)...")
print("⏳ Do not interrupt the runtime during training.")

In [ ]:
%cd /content/Retrieval-based-Voice-Conversion-WebUI

!python train_nsf_sim_cache_sid_load_pretrain.py \
  -e "{MODEL_NAME}" \
  -sr {SAMPLE_RATE} \
  -f0 1 \
  -bs {BATCH_SIZE} \
  -g 0 \
  -te {TOTAL_EPOCH} \
  -se {SAVE_EVERY_EPOCH} \
  -pg assets/pretrained_v2/f0G40k.pth \
  -pd assets/pretrained_v2/f0D40k.pth \
  -l 1 \
  -c 0 \
  -sw 1 \
  -v v2

print("\n🎉 Training complete!")

In [ ]:
# Build FAISS index (improves voice similarity)
print("📊 Building FAISS index...")

faiss_lines = [
    'import numpy as np, os, faiss',
    f'model_name  = "{MODEL_NAME}"',
    f'log_dir     = "logs/{MODEL_NAME}"',
    'feature_dir = os.path.join(log_dir, "3_feature256")',
    'if os.path.exists(feature_dir):',
    '    npys = []',
    '    for fname in sorted(os.listdir(feature_dir)):',
    '        if fname.endswith(".npy"):',
    '            npys.append(np.load(os.path.join(feature_dir, fname)))',
    '    if npys:',
    '        big_npy = np.concatenate(npys, axis=0)',
    '        n_ivf   = min(int(16 * np.sqrt(big_npy.shape[0])), big_npy.shape[0] // 39)',
    '        index   = faiss.index_factory(256, "IVF" + str(n_ivf) + ",Flat")',
    '        index.train(big_npy)',
    '        index.add(big_npy)',
    '        index_path = os.path.join(log_dir, "trained_IVF" + str(n_ivf) + "_Flat_nprobe_1_" + model_name + "_v2.index")',
    '        faiss.write_index(index, index_path)',
    '        print("Index saved:", index_path)',
    '    else:',
    '        print("No feature files found")',
    'else:',
    '    print("Feature directory not found:", feature_dir)',
]
with open('/tmp/make_faiss.py', 'w') as fp:
    fp.write('\n'.join(faiss_lines))

!python3 /tmp/make_faiss.py
print("✅ FAISS index creation complete!")

In [ ]:
# Save model to Google Drive (optional but recommended)
import shutil, os

MODEL_NAME      = "my_voice"  # <- Must match Step 2
DRIVE_SAVE_PATH = f"/content/drive/MyDrive/RVC_Models/{MODEL_NAME}"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

log_dir = f"logs/{MODEL_NAME}"
saved   = []

for f in os.listdir(log_dir):
    if f.endswith('.pth') or f.endswith('.index'):
        shutil.copy(os.path.join(log_dir, f), os.path.join(DRIVE_SAVE_PATH, f))
        saved.append(f)

print(f"✅ Saved to Google Drive: {DRIVE_SAVE_PATH}")
for s in saved:
    print(f"  📁 {s}")

---
## 🗣️ Step 5: Type English Text → Hear It in Your Voice!

RVC is a voice conversion model, so the pipeline is:  
**Edge-TTS** (generates a neutral English voice) → **RVC** (converts it into your voice)

Available English voices for Edge-TTS:
- `en-US-GuyNeural` — US English male
- `en-US-JennyNeural` — US English female
- `en-GB-RyanNeural` — British English male
- `en-GB-SoniaNeural` — British English female
- `en-AU-NatashaNeural` — Australian English female

In [ ]:
# Install Edge-TTS
!pip install -q edge-tts
print("✅ Edge-TTS installed")

In [ ]:
# List all available English voices
import asyncio, edge_tts

async def list_english_voices():
    voices = await edge_tts.list_voices()
    english = [v for v in voices if v['Locale'].startswith('en')]
    for v in english:
        print(f"  {v['ShortName']} - {v['FriendlyName']}")

print("🇬🇧 Available English voices:")
asyncio.run(list_english_voices())

In [ ]:
# Generate base English TTS
import asyncio, edge_tts

# ✏️ Edit these settings
TEXT        = "Hello! This is my voice speaking in English using RVC. Pretty amazing, right?"
BASE_VOICE  = "en-US-GuyNeural"       # <- Change voice here (male: GuyNeural, female: JennyNeural)
BASE_OUTPUT = "/content/base_tts.wav" # Intermediate file

async def generate_base_tts():
    communicate = edge_tts.Communicate(TEXT, BASE_VOICE)
    await communicate.save(BASE_OUTPUT)

print("🗣️ Generating base English TTS...")
print(f"   Text : {TEXT}")
print(f"   Voice: {BASE_VOICE}")
asyncio.run(generate_base_tts())
print(f"✅ Base audio saved: {BASE_OUTPUT}")

from IPython.display import Audio, display
print("\n🔊 Base voice (before conversion):")
display(Audio(BASE_OUTPUT))

In [ ]:
# Find the latest trained model files
import os, glob

MODEL_NAME  = "my_voice"  # <- Must match Step 2
log_dir     = f"logs/{MODEL_NAME}"
pth_files   = sorted(glob.glob(f"{log_dir}/*.pth"),   key=os.path.getmtime, reverse=True)
index_files = sorted(glob.glob(f"{log_dir}/*.index"), key=os.path.getmtime, reverse=True)

if pth_files:
    MODEL_PATH = pth_files[0]
    print(f"✅ Model file : {MODEL_PATH}")
else:
    print("❌ No trained model found. Please run Step 4 first.")

if index_files:
    INDEX_PATH = index_files[0]
    print(f"✅ Index file : {INDEX_PATH}")
else:
    INDEX_PATH = ""
    print("⚠️ No index file found (quality may be slightly lower)")

In [ ]:
# Convert to your voice using RVC!
import sys
sys.path.append('/content/Retrieval-based-Voice-Conversion-WebUI')

OUTPUT_PATH = "/content/my_voice_tts.wav"

!python infer_cli.py \
  --model_path "{MODEL_PATH}" \
  --index_path "{INDEX_PATH}" \
  --input_path "{BASE_OUTPUT}" \
  --output_path "{OUTPUT_PATH}" \
  --f0method "rmvpe" \
  --f0up_key 0 \
  --index_rate 0.75 \
  --protect 0.33

print("\n🎉 Voice conversion complete!")

In [ ]:
# Compare: before vs after
from IPython.display import Audio, display
import os

print("🔊 Base voice (before):")
display(Audio(BASE_OUTPUT))

print("\n🎙️ Your voice TTS (after):")
if os.path.exists(OUTPUT_PATH):
    display(Audio(OUTPUT_PATH))
    print(f"\n✅ Saved: {OUTPUT_PATH}")
else:
    print("❌ Output file was not created. Check for errors above.")

In [ ]:
# Download the output TTS file
from google.colab import files
import os

if os.path.exists(OUTPUT_PATH):
    print("📥 Downloading TTS file...")
    files.download(OUTPUT_PATH)
else:
    print("❌ No file to download.")

---
## 🔁 Quick Use — Change Text & Generate Again

After training is done, just run this single cell every time you want a new TTS!

In [ ]:
import asyncio, edge_tts, os
from IPython.display import Audio, display
from google.colab import files

# ✏️ Edit only this section!
TEXT        = "Welcome to the future of voice technology. This is my cloned voice speaking."  # <- Your text here
BASE_VOICE  = "en-US-GuyNeural"  # <- Voice: GuyNeural / JennyNeural / RyanNeural / SoniaNeural
F0_KEY      = 0     # Pitch shift (-12 ~ +12, 0 = no change)
INDEX_RATE  = 0.75  # Voice similarity (0~1, higher = more like your voice)

BASE_OUTPUT = "/content/base_tts.wav"
OUTPUT_PATH = "/content/my_voice_tts.wav"

# 1. Generate base TTS
async def gen_tts():
    await edge_tts.Communicate(TEXT, BASE_VOICE).save(BASE_OUTPUT)
asyncio.run(gen_tts())

# 2. Convert to your voice
os.system(
    'python infer_cli.py'
    f' --model_path "{MODEL_PATH}"'
    f' --index_path "{INDEX_PATH}"'
    f' --input_path "{BASE_OUTPUT}"'
    f' --output_path "{OUTPUT_PATH}"'
    f' --f0method rmvpe'
    f' --f0up_key {F0_KEY}'
    f' --index_rate {INDEX_RATE}'
    f' --protect 0.33'
)

# 3. Play result
print(f"📢 Text: {TEXT}")
print("🎙️ Your voice TTS:")
display(Audio(OUTPUT_PATH))

# 4. Download
files.download(OUTPUT_PATH)

---
## 📚 Troubleshooting

| Problem | Solution |
|---------|----------|
| CUDA out of memory | Reduce BATCH_SIZE to 4 or 5 |
| Voice sounds robotic | Adjust F0_KEY, train with more data |
| Too much noise | Improve recording quality, lower INDEX_RATE |
| Training is slow | Confirm runtime is set to T4 GPU |
| NameError: MODEL_NAME | Re-run the config cell at the top of that step |
| infer_cli.py not found | Run `!ls` to check available files |

## 💡 Tips for Better Quality
- Record in a **quiet room** with no echo or background noise
- Read **varied English content** — news articles, stories, conversations
- More epochs improve quality, but watch for overfitting (loss stops decreasing)
- **INDEX_RATE** between 0.6 and 0.8 is usually the sweet spot
- Use a **consistent tone and pace** across all recordings
